# AutoResolve: NLP-Based Automated Ticket Triage and Generative Resolution Agent
**Author:** Shaury Pratap Singh

**Course:** Introduction to NLP

## Abstract
This project focuses on building an end-to-end automated IT support ticket triage and resolution system, "AutoResolve." The objective is to accurately categorize user support requests into specific intents and automatically generate context-aware solutions. We compare traditional mathematical modeling (TF-IDF with Support Vector Machines) against a state-of-the-art Deep Learning architecture (DistilBERT) for intent classification. Finally, we extend the pipeline into a Retrieval-Augmented Generation (RAG) agent using a FAISS vector database and a 4-bit quantized Generative LLM to dynamically synthesize hallucination-free IT resolutions.

In [ ]:
# Install the necessary Hugging Face libraries
!pip install datasets pandas

# Import libraries
from datasets import load_dataset
import pandas as pd

print("Initializing Data Acquisition...")

# 1. Load the dataset directly from the Hugging Face hub
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# 2. Convert the Hugging Face Dataset object into a Pandas DataFrame
df = pd.DataFrame(dataset['train'])

# 3. Filter down to only the essential columns for our NLP pipeline
df = df[['instruction', 'intent']]
df.columns = ['text', 'label']

# 4. Display the shape and the first few rows
print(f"Dataset loaded successfully. Shape: {df.shape}")
display(df.head())

---
## 1. Linguistic Preprocessing & Exploratory Data Analysis
**Relevant Syllabus Concepts: Text Processing, N-gram Language Models**

Real-world enterprise data contains highly specific formatting. In the Bitext dataset, contextual anchors like `{{Order Number}}` are used. A standard preprocessing pipeline would strip these out as punctuation, destroying critical context.

Here, we implement an advanced pipeline using Regular Expressions and `spaCy`. We isolate the masked variables, lemmatize the standard text, and remove stop words. We then generate a Bigram frequency distribution to visually analyze the N-gram structure of our most common user queries.

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
import re

print("Initializing Advanced Preprocessing & EDA...")

# 1. Load spaCy's small English model (optimized for speed)
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

# 2. Custom text cleaning function protecting the {{Placeholders}}
def clean_text_fixed(text):
    # Target ONLY the text inside the placeholders to remove spaces
    def process_match(match):
        return "MASKED_" + match.group(1).replace(" ", "")

    text = re.sub(r'\{\{(.*?)\}\}', process_match, text)

    # Process through spaCy
    doc = nlp(text.lower())

    # Extract lemmatized tokens
    cleaned_tokens = [
        token.lemma_ for token in doc
        if (token.is_alpha or 'masked' in token.text) and not token.is_stop
    ]

    return " ".join(cleaned_tokens)

# 3. Apply the cleaning function
tqdm.pandas(desc="Cleaning text data")
df['cleaned_text'] = df['text'].progress_apply(clean_text_fixed)

# --- Exploratory Data Analysis (EDA) ---
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 12))

# Plot 1: Class Distribution
plt.subplot(2, 1, 1)
intent_counts = df['label'].value_counts()
sns.barplot(x=intent_counts.values, y=intent_counts.index, hue=intent_counts.index, palette="viridis", legend=False)
plt.title('Distribution of Customer Support Intents', fontsize=14, fontweight='bold')
plt.xlabel('Number of Tickets', fontsize=12)
plt.ylabel('Intent Category', fontsize=12)

# Plot 2: Top 20 Bigrams
plt.subplot(2, 1, 2)
vectorizer = CountVectorizer(ngram_range=(2, 2), max_features=20)
bigrams = vectorizer.fit_transform(df['cleaned_text'])
bigram_frequencies = sum(bigrams).toarray()[0]
bigram_df = pd.DataFrame({'Bigram': vectorizer.get_feature_names_out(), 'Frequency': bigram_frequencies})
bigram_df = bigram_df.sort_values(by='Frequency', ascending=False)

sns.barplot(x='Frequency', y='Bigram', data=bigram_df, hue='Bigram', palette="mako", legend=False)
plt.title('Top 20 Bigrams in Cleaned Ticket Descriptions', fontsize=14, fontweight='bold')
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Bigram', fontsize=12)

plt.tight_layout()
plt.show()

---
## 2. Baseline Mathematical Modeling
**Relevant Syllabus Concepts: Vector Semantics, Naive Bayes, Logistic Regression/SVM**

Before implementing deep learning, we must establish a mathematical baseline. We convert our cleaned text corpus into a sparse numerical matrix using **Term Frequency-Inverse Document Frequency (TF-IDF)**. This algorithm penalizes common words and boosts rare, highly descriptive tokens.

We train a Support Vector Machine (SVM) and a Multinomial Naive Bayes classifier on these vectors to see how well traditional frequency-based routing performs.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

print("Initializing Baseline Modeling...")

# Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Vector Semantics: TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Model 1: Linear Support Vector Machine
print("Training Linear Support Vector Machine...")
svm_model = LinearSVC(random_state=42, dual=False)
svm_model.fit(X_train_tfidf, y_train)

# Model 2: Multinomial Naive Bayes
print("Training Multinomial Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Evaluation
svm_preds = svm_model.predict(X_test_tfidf)
nb_preds = nb_model.predict(X_test_tfidf)

print("\n==========================================")
print("   SUPPORT VECTOR MACHINE PERFORMANCE")
print("==========================================")
print(classification_report(y_test, svm_preds))

print("\n==========================================")
print("      NAIVE BAYES PERFORMANCE")
print("==========================================")
print(classification_report(y_test, nb_preds))

---
## 3. Transformer Architecture Setup
**Relevant Syllabus Concepts: Transformers, Pretrained Language Models, Attention Mechanisms**

While the TF-IDF baseline performed exceptionally well, it relies entirely on rigid keyword matching. It ignores word order and contextual nuance. To build a robust system capable of handling sarcasm, typos, and convoluted grammar, we shift to a Transformer architecture (DistilBERT).

**Crucial Linguistic Shift:** Unlike the SVM, we feed the *raw, uncleaned text* into the Transformer. DistilBERT's bidirectional self-attention mechanisms require the original sentence structure, punctuation, and stop words to accurately compute the directional context of the query.

In [ ]:
!pip install transformers torch scikit-learn

import torch
from transformers import DistilBertTokenizerFast
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Initializing Transformer Data Setup...")

# Encode text labels into integers
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(df['label'])

# Train-Test Split (Using RAW text)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    encoded_labels.tolist(),
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

# Load DistilBERT Tokenizer
print("Downloading DistilBERT Tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenize the text
print("Tokenizing datasets...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Custom PyTorch Dataset Class
class TicketDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Instantiate datasets
train_dataset = TicketDataset(train_encodings, train_labels)
val_dataset = TicketDataset(val_encodings, val_labels)

print("\nPyTorch Datasets created successfully!")

---
## 4. Model Fine-Tuning & Transfer Learning
**Relevant Syllabus Concepts: Fine-Tuning and Masked Language Models**

We initialize a pre-trained `distilbert-base-uncased` model and replace its generic language head with a custom classification head mapped to our specific intent categories. We fine-tune the network weights over 3 epochs, utilizing weight decay and warmup steps to prevent the model from overfitting to the synthetic constraints of the Bitext dataset.

In [ ]:
import numpy as np
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Initializing DistilBERT Model Architecture...")

num_labels = len(label_encoder.classes_)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\nCommencing Transformer Fine-Tuning...")
trainer.train()

print("\nSaving finalized model artifacts...")
trainer.save_model("./autoresolve_distilbert_final")
tokenizer.save_pretrained("./autoresolve_distilbert_final")

---
## 5. Evaluation & Qualitative Error Analysis
**Relevant Syllabus Concepts: LLM Toxicity and Bias / Failure Modes**

Accuracy metrics alone do not prove understanding. To validate our deep learning approach, we generate a Confusion Matrix to map our predictions against true labels.

More importantly, we isolate the instances where the model made high-confidence errors (predicting the wrong intent with >99% mathematical certainty). Analyzing these specific semantic failures allows us to understand the limitations of the attention mechanism, particularly regarding out-of-vocabulary misspellings and lexical ambiguity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn.functional as F

print("Initializing Advanced Evaluation & Error Analysis...")

# 1. Generate predictions
raw_predictions = trainer.predict(val_dataset)

# 2. Extract probabilities and confidence scores
logits = torch.tensor(raw_predictions.predictions)
probabilities = F.softmax(logits, dim=-1)
confidence_scores = torch.max(probabilities, dim=-1).values.numpy()
predicted_classes = np.argmax(raw_predictions.predictions, axis=-1)
true_classes = raw_predictions.label_ids

# 3. Create Confusion Matrix
plt.figure(figsize=(18, 14))
cm = confusion_matrix(true_classes, predicted_classes)
class_names = label_encoder.inverse_transform(np.arange(num_labels))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('AutoResolve DistilBERT Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Intent', fontsize=12)
plt.ylabel('Actual Intent', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Extract High-Confidence Mistakes
print("\n=======================================================")
print("  QUALITATIVE ERROR ANALYSIS: HIGH-CONFIDENCE MISTAKES")
print("=======================================================")

error_df = pd.DataFrame({
    'Text': val_texts,
    'True_Label': label_encoder.inverse_transform(true_classes),
    'Predicted_Label': label_encoder.inverse_transform(predicted_classes),
    'Confidence': confidence_scores
})

mistakes = error_df[error_df['True_Label'] != error_df['Predicted_Label']]
worst_mistakes = mistakes.sort_values(by='Confidence', ascending=False).head(5)

if worst_mistakes.empty:
    print("Incredible. The model made zero mistakes on the validation set.")
else:
    for index, row in worst_mistakes.iterrows():
        print(f"\n[Raw Text]: {row['Text']}")
        print(f"-> True Intent:      {row['True_Label']}")
        print(f"-> Predicted Intent: {row['Predicted_Label']} (Confidence: {row['Confidence']:.4f})")

---
## 6. Retrieval-Augmented Generation (RAG) Agent Integration
**Relevant Syllabus Concepts: Question Answering (Week 13), LLM Agents (Week 14)**

Intent classification is only the first half of ticket triage; the second half is automated resolution. To elevate this pipeline into a true LLM Agent, we integrate a Retrieval-Augmented Generation (RAG) architecture.

Instead of relying on a generative model's internal, potentially hallucinated parametric memory, we embed a structured Knowledge Base (simulating an enterprise PostgreSQL/Vector database) using Sentence Transformers. We then use **Cosine Similarity** to retrieve the most relevant IT documentation:

$$\text{Cosine Similarity} = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

Finally, we pass the retrieved document and the user's query into a heavily quantized generative LLM (Llama 3 8B) to synthesize a coherent, contextually grounded resolution.

In [ ]:
!pip install sentence-transformers faiss-cpu accelerate bitsandbytes

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Initializing AutoResolve Knowledge Base and Vector Database...")

# 1. Create a mock Enterprise IT Knowledge Base
# In production, this would be your PostgreSQL database. For Colab, we use a list.
knowledge_base = [
    "Refund Policy: Customers are entitled to a full refund within 30 days of purchase. To process, verify the order number and issue the refund to the original payment method.",
    "Order Tracking: To locate an order, query the shipping database using the 10-digit order number. If the status is 'Dispatched', provide the user with the carrier tracking link.",
    "Password Recovery: If a user cannot log in, send a secure password reset link to their registered email address. Ensure they check their spam folder.",
    "Payment Issues: If a transfer or payment fails, verify if the credit card is expired or if the anti-fraud system flagged the transaction. Recommend trying a different payment method."
]

# 2. Initialize the Embedding Model (Converts text into mathematical vectors)
print("Loading Sentence Transformer...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Embed the Knowledge Base
kb_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True)

# 4. Initialize FAISS (Facebook AI Similarity Search) for fast vector retrieval
# We use an L2 distance index (mathematically related to cosine similarity for normalized vectors)
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)

print(f"Successfully embedded and indexed {index.ntotal} IT resolution documents.")

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder

print("Flushing GPU Memory...")

# 1. Nuke the old training artifacts from the GPU to free up VRAM
try:
    del trainer
    del model
except NameError:
    pass # If they are already gone, ignore
gc.collect()
torch.cuda.empty_cache()
print(f"GPU Memory Cleared. Available VRAM is ready.")

# 2. Reload the Fine-Tuned DistilBERT
print("Reloading AutoResolve Intent Classifier...")
distil_tokenizer = DistilBertTokenizerFast.from_pretrained("./autoresolve_distilbert_final")
distil_model = DistilBertForSequenceClassification.from_pretrained("./autoresolve_distilbert_final").to('cuda')

# Ensure the label encoder is active
label_encoder = LabelEncoder()
label_encoder.fit(df['label'])

# 3. Configure Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 4. Load a lighter, highly capable LLM (Qwen 2.5 1.5B) to guarantee it fits in Colab
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading Generative LLM ({model_id})...")
llama_tokenizer = AutoTokenizer.from_pretrained(model_id)
llama_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. The AutoResolve End-to-End Pipeline Function
def autoresolve_agent(user_query):
    print(f"\n=====================================")
    print(f"NEW TICKET: {user_query}")
    print(f"=====================================")

    # Step A: Intent Classification
    inputs = distil_tokenizer(user_query, return_tensors="pt", truncation=True, padding=True).to('cuda')
    with torch.no_grad():
        logits = distil_model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    predicted_intent = label_encoder.inverse_transform([predicted_class_id])[0]
    print(f"-> [Classification Phase] Intent Identified: {predicted_intent}")

    # Step B: RAG Retrieval
    query_vector = embedder.encode([user_query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, 1)
    retrieved_doc = knowledge_base[indices[0][0]]
    print(f"-> [Retrieval Phase] KB Document Accessed: {retrieved_doc[:75]}...")

    # Step C: LLM Generation
    prompt = f"""<|im_start|>system
You are AutoResolve, an IT support agent. Answer the user's query using ONLY the provided IT Document. Be polite, concise, and professional.<|im_end|>
<|im_start|>user
User Query: {user_query}
IT Document: {retrieved_doc}<|im_end|>
<|im_start|>assistant
"""

    inputs = llama_tokenizer(prompt, return_tensors="pt").to('cuda')

    # Generate the text
    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1,
        pad_token_id=llama_tokenizer.eos_token_id
    )

    # Decode and format the output
    full_response = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Isolate just the assistant's generated text
    final_answer = full_response.split("assistant\n")[-1].strip()

    print(f"\n-> [Generation Phase] Final Agent Response:\n{final_answer}")

# Test the fully integrated pipeline!
autoresolve_agent("am I entitled to a reimbursement?")